# Semana 8: Evaluación y Validación de Modelos
## Notebook de Ejercicios (NB2) – Comparación de Modelos en Predicción de Churn

**Propósito:** Consolidar las técnicas de evaluación y validación aplicándolas a un problema real de clasificación: predicción de abandono de clientes (churn).

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Evaluar múltiples modelos (regresión logística, SVM, Random Forest, XGBoost) con validación cruzada.
- Dibujar curvas ROC y comparar AUC.
- Seleccionar los mejores hiperparámetros para cada modelo.
- Elegir el modelo final justificando con métricas y consideraciones de negocio.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, roc_auc_score, classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline

# XGBoost
import xgboost as xgb

# Para búsqueda aleatoria
from scipy.stats import randint, uniform

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Carga del Dataset: Telco Customer Churn

El dataset de **Telco Customer Churn** contiene información sobre clientes de una compañía de telecomunicaciones y si abandonaron o no el servicio.

**Características:**
- Datos demográficos (género, edad, etc.)
- Datos del servicio (tipo de contrato, método de pago, etc.)
- Datos de uso (cargos mensuales, totales, etc.)
- Variable objetivo: `Churn` (Yes/No)

Cargamos desde una URL pública.

In [ ]:
# URL del dataset (Kaggle Telco Customer Churn)
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'

try:
    df = pd.read_csv(url)
    print("Dataset cargado correctamente.")
except:
    print("No se pudo cargar desde URL. Usando datos locales de muestra...")
    # Datos de muestra mínimos para que el notebook no falle
    df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/telco_churn_sample.csv')

print(f"Dimensiones del dataset: {df.shape}")
print("\nPrimeras 5 filas:")
df.head()

In [ ]:
# Información general
df.info()

In [ ]:
# Distribución de la variable objetivo
if 'Churn' in df.columns:
    print("Distribución de Churn:")
    print(df['Churn'].value_counts())
    print(f"\nPorcentaje de abandono: {df['Churn'].value_counts(normalize=True)[1]*100:.2f}%")

    plt.figure(figsize=(6, 4))
    sns.countplot(data=df, x='Churn')
    plt.title('Distribución de Churn')
    plt.show()

---
## 2. Preprocesamiento de Datos

El dataset contiene variables numéricas y categóricas. También hay valores nulos en algunas columnas (como `TotalCharges`).

In [ ]:
# Verificamos nulos
print("Valores nulos por columna:")
print(df.isnull().sum())

# Para TotalCharges, puede ser que sea string y tenga espacios
if 'TotalCharges' in df.columns:
    # Convertimos a numérico, forzando errores a NaN
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    # Imputamos con la mediana
    df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Convertimos variable objetivo a binaria
if 'Churn' in df.columns:
    df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Identificamos columnas numéricas y categóricas
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'Churn' in numeric_cols:
    numeric_cols.remove('Churn')
    
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"\nColumnas numéricas: {numeric_cols}")
print(f"Columnas categóricas: {categorical_cols}")

# Codificamos variables categóricas con one-hot encoding
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separamos features y target
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

print(f"\nDimensiones finales: {X.shape}")
X.head()

In [ ]:
# Dividimos en entrenamiento y prueba (estratificado)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Tamaño entrenamiento: {X_train.shape}")
print(f"Tamaño prueba: {X_test.shape}")
print(f"Proporción de churn en train: {y_train.mean():.3f}")
print(f"Proporción de churn en test: {y_test.mean():.3f}")

# Escalamos características (importante para regresión logística y SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

---
## 3. Evaluación de Modelos Base con Validación Cruzada

Evaluamos cuatro modelos con sus parámetros por defecto usando validación cruzada (5-fold).

In [ ]:
models = {
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
    'SVM (RBF)': SVC(probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(eval_metric='logloss', random_state=42)
}

results_base = []

for name, model in models.items():
    # Para SVM y Regresión Logística usamos datos escalados
    if name in ['Regresión Logística', 'SVM (RBF)']:
        X_cv = X_train_scaled
    else:
        X_cv = X_train
    
    scores = cross_val_score(model, X_cv, y_train, cv=5, scoring='roc_auc')
    results_base.append({
        'Modelo': name,
        'AUC CV medio': scores.mean(),
        'AUC CV std': scores.std()
    })
    print(f"{name}: AUC = {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

df_results_base = pd.DataFrame(results_base)
print("\n=== Resultados Base ===")
df_results_base

---
## 4. Optimización de Hiperparámetros

Seleccionamos los mejores hiperparámetros para cada modelo usando GridSearchCV o RandomizedSearchCV según corresponda.

### 4.1. Regresión Logística

In [ ]:
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

lr = LogisticRegression(max_iter=1000, random_state=42)
grid_lr = GridSearchCV(lr, param_grid_lr, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_lr.fit(X_train_scaled, y_train)

print(f"Mejores parámetros LR: {grid_lr.best_params_}")
print(f"Mejor AUC CV: {grid_lr.best_score_:.4f}")

### 4.2. SVM

In [ ]:
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.01, 0.1, 1, 'scale', 'auto'],
    'kernel': ['rbf']
}

svm = SVC(probability=True, random_state=42)
grid_svm = GridSearchCV(svm, param_grid_svm, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_svm.fit(X_train_scaled, y_train)

print(f"Mejores parámetros SVM: {grid_svm.best_params_}")
print(f"Mejor AUC CV: {grid_svm.best_score_:.4f}")

### 4.3. Random Forest

In [ ]:
param_dist_rf = {
    'n_estimators': randint(50, 300),
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2']
}

rf = RandomForestClassifier(random_state=42)
random_rf = RandomizedSearchCV(rf, param_dist_rf, n_iter=30, cv=5, scoring='roc_auc', 
                                random_state=42, n_jobs=-1, verbose=1)
random_rf.fit(X_train, y_train)

print(f"Mejores parámetros RF: {random_rf.best_params_}")
print(f"Mejor AUC CV: {random_rf.best_score_:.4f}")

### 4.4. XGBoost

In [ ]:
param_dist_xgb = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 0.5)
}

xgb_model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
random_xgb = RandomizedSearchCV(xgb_model, param_dist_xgb, n_iter=30, cv=5, scoring='roc_auc',
                                 random_state=42, n_jobs=-1, verbose=1)
random_xgb.fit(X_train, y_train)

print(f"Mejores parámetros XGBoost: {random_xgb.best_params_}")
print(f"Mejor AUC CV: {random_xgb.best_score_:.4f}")

---
## 5. Evaluación en Test y Curvas ROC

Evaluamos los modelos optimizados en el conjunto de prueba y dibujamos sus curvas ROC.

In [ ]:
best_models = {
    'Regresión Logística': grid_lr.best_estimator_,
    'SVM (RBF)': grid_svm.best_estimator_,
    'Random Forest': random_rf.best_estimator_,
    'XGBoost': random_xgb.best_estimator_
}

plt.figure(figsize=(10, 8))

for name, model in best_models.items():
    # Predecimos probabilidades
    if name in ['Regresión Logística', 'SVM (RBF)']:
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Aleatorio')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curvas ROC - Comparación de Modelos')
plt.legend()
plt.grid(True)
plt.show()

## 6. Métricas Detalladas por Modelo

In [ ]:
results_test = []

for name, model in best_models.items():
    # Predecimos
    if name in ['Regresión Logística', 'SVM (RBF)']:
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
    
    results_test.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_proba)
    })

df_results_test = pd.DataFrame(results_test)
print("=== Resultados en Test ===")
df_results_test.round(4)

### 6.1. Matrices de Confusión

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, (name, model) in enumerate(best_models.items()):
    if name in ['Regresión Logística', 'SVM (RBF)']:
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)
    
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i])
    axes[i].set_title(f'Matriz de Confusión - {name}')
    axes[i].set_xlabel('Predicho')
    axes[i].set_ylabel('Real')

plt.tight_layout()
plt.show()

---
## 7. Selección del Modelo Final

Basado en los resultados y consideraciones de negocio, elegimos el modelo más adecuado.

In [ ]:
# Mostramos el mejor modelo por AUC
best_auc_idx = df_results_test['AUC'].idxmax()
best_model_name = df_results_test.loc[best_auc_idx, 'Modelo']
best_auc = df_results_test.loc[best_auc_idx, 'AUC']

print(f"Mejor modelo por AUC: {best_model_name} con AUC = {best_auc:.4f}")

# Consideraciones de negocio
print("\n=== ANÁLISIS PARA SELECCIÓN ===")
print("""
En un problema de predicción de churn, las métricas más relevantes suelen ser:

- **Recall (Sensibilidad)**: Queremos detectar la mayor cantidad posible de clientes que van a abandonar (evitar falsos negativos). Un cliente que abandona y no fue detectado es una pérdida de ingresos.
- **Precisión**: Los falsos positivos (clientes que pensamos que abandonarán pero no lo hacen) pueden llevar a ofrecer incentivos innecesarios, lo que tiene un costo.
- **AUC**: Mide la capacidad discriminativa global, independiente del umbral.

Dependiendo de la estrategia de retención:
- Si los incentivos son caros, priorizamos **precisión**.
- Si el costo de perder un cliente es muy alto, priorizamos **recall**.
- A menudo se busca un equilibrio usando **F1** o ajustando el umbral.
""")

# Mostramos las métricas ordenadas por recall
print("\nModelos ordenados por Recall:")
print(df_results_test.sort_values('Recall', ascending=False).round(4))

# Decisión final
print("\n=== DECISIÓN FINAL ===")
print(f"Basado en el análisis, seleccionamos {best_model_name} como modelo final.")
print("Justificación: Ofrece el mejor equilibrio entre AUC, recall y precisión, "
      "y es suficientemente interpretable para explicar decisiones de negocio.")

## 8. Importancia de Características (opcional)

Si el modelo seleccionado es Random Forest o XGBoost, podemos analizar las variables más importantes.

In [ ]:
if best_model_name in ['Random Forest', 'XGBoost']:
    model = best_models[best_model_name]
    importances = model.feature_importances_
    feature_names = X.columns
    
    imp_df = pd.DataFrame({
        'Característica': feature_names,
        'Importancia': importances
    }).sort_values('Importancia', ascending=False).head(15)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=imp_df, x='Importancia', y='Característica')
    plt.title(f'Top 15 Características más Importantes - {best_model_name}')
    plt.tight_layout()
    plt.show()
else:
    print(f"El modelo {best_model_name} no proporciona importancia de características directamente.")

---
## 9. Conclusiones

En este notebook hemos aplicado un flujo completo de evaluación y selección de modelos:

✔️ **Evaluación base**: Comparamos 4 modelos con validación cruzada.
✔️ **Optimización de hiperparámetros**: Ajustamos cada modelo con GridSearchCV o RandomizedSearchCV.
✔️ **Curvas ROC y AUC**: Visualizamos y comparamos el rendimiento.
✔️ **Métricas detalladas**: Accuracy, precisión, recall, F1 y AUC.
✔️ **Selección final**: Elegimos el mejor modelo según métricas y consideraciones de negocio.

**Lección clave**: No existe un modelo universalmente mejor; la elección depende del problema, las métricas y el contexto de negocio. La validación rigurosa es esencial para tomar decisiones informadas.

---
**Fin del Notebook de Ejercicios - Semana 8**